# Regularized linear regression: Ridge, Lasso, Elastic Net

**Supervised learning — regression.** We compare **ordinary least squares** (linear regression) with **L2** (Ridge), **L1** (Lasso), and **Elastic Net** (L1+L2) on the California housing dataset.

- **Ridge** shrinks coefficients; good when many correlated features.
- **Lasso** promotes sparsity (some coefficients exactly zero).
- **Elastic Net** mixes both.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "Lasso": Lasso(alpha=0.01, random_state=42, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42, max_iter=10000),
}

rows = []
for name, est in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", est)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({
        "model": name,
        "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
        "r2": float(r2_score(y_test, pred)),
    })

df = pd.DataFrame(rows).sort_values("rmse")
print(df.to_string(index=False))

coef_model = Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.01, random_state=42, max_iter=10000))])
coef_model.fit(X_train, y_train)
coefs = coef_model.named_steps["model"].coef_
plt.figure(figsize=(8, 4))
plt.barh(X.columns, coefs)
plt.title("Lasso coefficients (after scaling)")
plt.tight_layout()
plt.show()

## Try this

- Tune `alpha` with `GridSearchCV` on Ridge/Lasso.
- Compare learning curves as you vary training set size.